In [6]:
import time
import numpy as np
import pytest
import skfuzzy as fuzzy

In [7]:
try:
    from src.engine import construir_motor_inferencia
    from src.model import criar_sistema_fuzzy
except ImportError:
    from engine import construir_motor_inferencia
    from model import criar_sistema_fuzzy


@pytest.fixture
def simulador():
    return construir_motor_inferencia()


def inferir(sim, impacto, explorabilidade):
    sim.input['impacto'] = impacto
    sim.input['explorabilidade'] = explorabilidade
    sim.compute()
    return sim.output['urgencia']


# ------------------------------ MODELO ----------------------------------------
def test_universos_do_discurso():
    impacto, explor, urgencia = criar_sistema_fuzzy()
    assert (impacto.universe.min(), impacto.universe.max()) == (0, 100)
    assert (explor.universe.min(), explor.universe.max()) == (0, 10)
    assert (urgencia.universe.min(), urgencia.universe.max()) == (0, 100)


def test_quantidade_de_conjuntos_fuzzy():
    impacto, explor, urgencia = criar_sistema_fuzzy()
    assert len(impacto.terms) == 3
    assert len(explor.terms) == 3
    assert len(urgencia.terms) == 4


def test_pertinencia_no_pico_do_impacto_medio():
    impacto, _, _ = criar_sistema_fuzzy()
    grau = fuzzy.interp_membership(impacto.universe, impacto['medio'].mf, 50)
    assert grau == pytest.approx(1.0)


def test_pertinencia_na_fronteira_de_impacto():
    # Em 30: baixo = (40-30)/40... trimf [0,0,40] -> 0.25 ; medio = (30-20)/30 = 0.333
    impacto, _, _ = criar_sistema_fuzzy()
    baixo = fuzzy.interp_membership(impacto.universe, impacto['baixo'].mf, 30)
    medio = fuzzy.interp_membership(impacto.universe, impacto['medio'].mf, 30)
    assert baixo == pytest.approx(0.25)
    assert medio == pytest.approx(1 / 3, abs=1e-3)


# ------------------------------ INFERÊNCIA ------------------------------------
def test_minimo_absoluto_gera_urgencia_baixa(simulador):
    assert inferir(simulador, 0.0, 0.0) == pytest.approx(10.0, abs=0.5)


def test_maximo_absoluto_gera_urgencia_critica(simulador):
    res = inferir(simulador, 100.0, 10.0)
    assert res == pytest.approx(92.22, abs=0.5)
    assert res > 80.0


def test_impacto_medio_moderado(simulador):
    assert inferir(simulador, 50.0, 5.0) == pytest.approx(76.67, abs=0.5)


def test_transicao_baixo_impacto_alta_explorabilidade(simulador):
    res = inferir(simulador, 10.0, 9.0)
    assert 40.0 < res < 55.0


def test_impacto_alto_independe_da_explorabilidade(simulador):
    # Regras 7, 8 e 9 mapeiam tudo para 'critica'
    r_baixa = inferir(simulador, 100.0, 0.0)
    r_alta = inferir(simulador, 100.0, 10.0)
    assert r_baixa == pytest.approx(r_alta, abs=0.5)


def test_monotonicidade_em_relacao_ao_impacto(simulador):
    valores = [inferir(simulador, i, 5.0) for i in (0, 25, 50, 75, 100)]
    assert valores == sorted(valores)


@pytest.mark.parametrize("impacto,explor", [(0, 0), (37, 3), (50, 5), (88, 7), (100, 10)])
def test_saida_sempre_dentro_do_universo(simulador, impacto, explor):
    res = inferir(simulador, impacto, explor)
    assert 0.0 <= res <= 100.0


# ------------------------------ LATÊNCIA --------------------------------------
def test_latencia_media_da_inferencia(simulador):
    inferir(simulador, 50, 5)  # aquecimento
    n = 50
    inicio = time.perf_counter()
    for i in range(n):
        inferir(simulador, (i * 2) % 101, (i % 11))
    media_ms = (time.perf_counter() - inicio) / n * 1000
    print(f"\n[LATÊNCIA] média por inferência: {media_ms:.2f} ms ({n} execuções)")
    assert media_ms < 50.0


# ------------------------------ SAÍDAS (PNG / CONSOLE) ------------------------
def test_gera_grafico_de_pertinencia(tmp_path, monkeypatch):
    try:
        from src.model import gerar_graficos_pertinencia
    except ImportError:
        from model import gerar_graficos_pertinencia
    monkeypatch.chdir(tmp_path)
    gerar_graficos_pertinencia()
    assert (tmp_path / "results" / "membership_functions.png").stat().st_size > 0


def test_gera_superficie_3d(simulador, tmp_path, monkeypatch):
    try:
        from src.engine import gerar_superficie_3d
    except ImportError:
        from engine import gerar_superficie_3d
    monkeypatch.chdir(tmp_path)
    gerar_superficie_3d(simulador)
    assert (tmp_path / "results" / "superficie_decisao.png").stat().st_size > 0


def test_simulacoes_de_console(simulador, capsys):
    try:
        from src.engine import simular_casos_teste
    except ImportError:
        from engine import simular_casos_teste
    simular_casos_teste(simulador)
    saida = capsys.readouterr().out
    assert "Mínimo Absoluto" in saida and "Máximo Absoluto" in saida

ModuleNotFoundError: No module named 'engine'